In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import json
from langchain_core.documents import Document

def load_docs(path, source):
    with open(path) as f:
        rows = json.load(f)
    docs = []
    for r in rows:
        meta = r.get('metadata', {})
        pages = meta.get('page', [])
        headings = meta.get('headings', [])
        docs.append(Document(
            page_content = r['text'],
            metadata={
                'chunk_id': r['chunk_id'],
                'source': source,
                'pages': ','.join(map(str, pages)),
                'headings': ' > '.join(headings)
            }
        ))
    return docs

docs = (
    load_docs('output/iam_chunks.json', 'iam') +
    load_docs('output/sagemaker_chunks.json', 'sagemaker') +
    load_docs('output/bedrock_chunks.json', 'bedrock')
)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = 'BAAI/bge-base-en-v1.5',
    model_kwargs= {'device': 'cuda'},
    encode_kwargs = {'normalize_embeddings': True, 'batch_size': 256}
)

In [ ]:
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

vectorstore = Chroma.from_documents(
    documents = docs,
    embedding = embeddings,
    persist_directory='storage/chroma',
    collection_name = 'aws_docs'
)
dense_retriever = vectorstore.as_retriever(search_kwargs={'k':30})

# Sparse retriever
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 30

# Hybird ensembling fusing dense and sparse retrievals
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

In [ ]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker

cross_encoder = HuggingFaceCrossEncoder(
    model_name = 'BAAI/bge-reranker-v2-m3',
    model_kwargs= {'device': 'cuda'}
)
reranker = CrossEncoderReranker(model=cross_encoder, top_n=5)
retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid_retriever
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    out = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get('source', '?')
        page = d.metadata.get('page', '')
        out.append(f'[{i}] ({src} p{page})\n{d.page_content}')
    return '\n\n'.join(out)

SYSTEM = """You are an AWS documentation assistant. Answer using ONLY the provided context.

Rules:
- Answer in 1-2 sentences. Nothing extra.
- Use only what the question asks for. No background, no "Key Context", no extra examples.
- Preserve exact tokens verbatim: service prefixes, ARNs, API names, policy JSON. Do not reformat or generalize them.
- Cite sources inline as [n] matching the context blocks.
- If the answer is not present in the context, reply with EXACTLY this and nothing else: "Not found in the provided AWS docs."
- Do NOT supply an answer from your own knowledge. If it is not in the context, it is "Not found" — never both.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM),
    ('human', 'Context:\n{context}\n\nQuestion: {question}')
])


In [ ]:
from langchain_community.llms import Ollama
from langchain_core.runnables import RunnablePassthrough

llm = Ollama(model="qwen2.5:7b", temperature=0, num_predict=256)

chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print(chain.invoke("What is the service prefix for AWS Certificate Manager?"))

In [ ]:
# from datasets import Dataset
# from ragas import evaluate
# from ragas.metrics import context_recall, context_precision, faithfulness, answer_relevancy
# from ragas.llms import LangchainLLMWrapper
# from ragas.embeddings import LangchainEmbeddingsWrapper
# from ragas.run_config import RunConfig

# judge_llm = LangchainLLMWrapper(Ollama(model="qwen2.5:7b", temperature=0))
# judge_emb = LangchainEmbeddingsWrapper(embeddings)

# golden = json.load(open('eval/golden_set.json'))

# rows = []
# for item in golden:
#     q = item['question']
#     docs = retriever.invoke(q)
#     answer = chain.invoke(q)
#     rows.append({
#         'question': q,
#         'answer': answer,
#         'contexts': [d.page_content for d in docs],
#         'ground_truth': item['ground_truth']
#     })

# dataset = Dataset.from_list(rows)
# result = evaluate(
#     dataset,
#     metrics = [context_recall, context_precision, faithfulness, answer_relevancy],
#     llm = judge_llm,
#     embeddings = judge_emb,
#     run_config=RunConfig(max_workers=1, timeout=300),
# )
# print(result)
# result.to_pandas().to_csv('eval/eval_results.csv', index=False)

In [ ]:
# History
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# rewrite follow-up using history
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given chat history and latest question, rewrite it as a standalone question. "
               "Resolve pronouns/references. Do NOT answer. Return only the rewritten question. "
               "If already standalone, return as-is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
history_retriever = create_history_aware_retriever(llm, retriever, contextualize_prompt)

# answer from retrieved docs and history
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM + "\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
qa_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_retriever, qa_chain)

In [ ]:
history = []
def ask(q):
    out = rag_chain.invoke({"input": q, "chat_history": history})
    history.extend([("human", q), ("ai", out["answer"])])
    return out["answer"]

ask("What is the ARN format for a SageMaker labeling job?")
ask("What about a model?")